scratchpad for byonoy movements and deck setup

In [1]:
import datetime
import logging
import time

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

simulation = True  # True or False

protocol_mode = "simulation" if simulation else "execution"

run_identifier = "nb5"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)

pylabrobot.verbose(True, level=logging.INFO)

print(f"logging to {log_path}")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

2026-09-28 01:39:04,898 - pylabrobot - INFO - --- nb5 (simulation) ---


logging to _logs/simulation/nb5_20260928-013904.log


In [2]:
from pylabrobot.hamilton.prep import Prep
from pylabrobot.resources.hamilton import PrepDeck

HOST = "192.168.100.102"  # as in the Prep demo notebook; ignored in simulation
PORT = 2000

deck = PrepDeck(with_core_grippers=True)

prep = Prep(
    deck=deck,
    simulation=simulation,
    host=HOST,
    port=PORT,
    )

await prep.setup(smart=True, force_initialize=False)

# await prep.lights.disco(duration=15)  # easter egg: random colours, then the colour it found
# await prep.lights.turn_off()
await prep.lights.turn_on()

2026-09-28 01:39:05,236 - pylabrobot.hamilton.prep.driver.master - INFO - tips held at setup: none
2026-09-28 01:39:05,237 - pylabrobot.hamilton.prep.driver.master - WARNING - the 8-channel head is not raised at setup: no move of its Z alone is known
2026-09-28 01:39:05,238 - pylabrobot.hamilton.prep.driver.master - INFO - [Hamilton Prep] Connected on simulation (no link)

  Serial: PRPBC1317

  Firmware: MLPrep Runtime V3.0.20.675 99020-02 Rev N

  Configuration: 
 - enclosure installed,  - safe speeds off,
 - traverse height 167.5 mm
  Deck: x 0.00 to 299.00 mm, y -9.00 to 385.00 mm, z 19.50 to 167.50 mm; 6 sites, 3 waste sites
  Pipettes: 2, v2 aspirate/dispense
    channel 0 (rear): firmware Channel1ml Runtime V1.6.2.270 98228-02 Rev U, x 0.45 to 299.45 mm, y 0.00 to 385.00 mm, z 19.50 to 167.50 mm
    channel 1 (front): firmware Channel1ml Runtime V1.6.2.270 98228-02 Rev U, x 0.45 to 299.45 mm, y -9.00 to 376.00 mm, z 19.50 to 167.50 mm
  8-channel head: installed
  heater shaker:

In [3]:
# await prep.pipettes.pick_up_tips(
#     [prep.get_resource("Prep_teaching_needle")],
#     # ntr_50ul_00["A3:B3"],
#     use_channels=[1]
#     )


In [4]:
# # probe_z_using_clld after the one-move change: two detections on the calibration block, then a
# # miss over the deck opening at (200, 200), which must come back to its start.
# channel_idx = 1
# block_top = prep.deck.calibration_block.get_location_wrt(deck, "c", "c", "t")
# results = {}

# await prep.pipettes.move_to_safe_z()
# await prep.pipettes.move_to_xy_positions(x=block_top.x, ys={channel_idx: block_top.y}, make_space=True)
# for post in (2.0, 0.0):
#     z = await prep.pipettes.probe_z_using_clld(
#         channel_idx=channel_idx,
#         search_start_position=block_top.z + 10,
#         search_end_position=block_top.z - 1,
#         search_speed=5,
#         post_detection_distance=post,
#     )
#     ended_z = (await prep.pipettes.request_locations())[channel_idx].z
#     results[f"block, post {post}"] = (z, ended_z)

# await prep.pipettes.move_to_safe_z()
# await prep.pipettes.move_to_xy_positions(x=200.0, ys={channel_idx: 199.99}, make_space=True)
# z = await prep.pipettes.probe_z_using_clld(
#     channel_idx=channel_idx,
#     search_start_position=32.0,
#     search_end_position=21.0,
#     search_speed=5,
# )
# ended_z = (await prep.pipettes.request_locations())[channel_idx].z
# results["opening, miss"] = (z, ended_z)

# await prep.pipettes.move_to_safe_z()
# results

In [5]:
# await prep.lights.turn_off()

In [6]:
# await prep.pipettes.pick_up_tips([prep.get_resource("Prep_teaching_needle")], use_channels=[0])
# print(await prep.pipettes.request_locations())
# await prep.pipettes.drop_tips([prep.get_resource("Prep_teaching_needle")], use_channels=[0])

In [ ]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

facility = Facility(name="facility", size_x=1_00, size_y=1_000, size_z=0)

bench = Resource(name="bench", size_x=1_000, size_y=700, size_z=1_000)
facility.assign_child_resource(bench, location=Coordinate(0, 300, 0))

bench.assign_child_resource(prep, location=Coordinate(10, 10, bench.get_size_z()))

viewer = Viewer3D(facility, name="v1_prep_manual.ipynb")
await viewer.start()

viewer on http://127.0.0.1:1339/#token=IRVtvFx2D06MOPFsjaxHPYMvytSh4gZjOBfyOacodSo  (websocket 2123)


viewer: a browser connected, drawing with WebGPU


## Where the channels stand

In [8]:
await prep.pipettes.request_locations()  # each channel's reference point, in deck mm

[Coordinate(x=289.49, y=365.01, z=167.5),
 Coordinate(x=289.49, y=345.01, z=167.5)]

In [9]:
await prep.pipettes.request_x_position()

289.49

In [10]:
await prep.pipettes.request_y_positions()

[365.01, 345.01]

In [11]:
await prep.pipettes.request_z_positions()

[167.5, 167.5]

In [12]:
await prep.pipettes.request_channel_bounds()

[{'x_min': 0.45,
  'x_max': 299.45,
  'y_min': 0.0,
  'y_max': 385.0,
  'z_min': 19.5,
  'z_max': 167.5},
 {'x_min': 0.45,
  'x_max': 299.45,
  'y_min': -9.0,
  'y_max': 376.0,
  'z_min': 19.5,
  'z_max': 167.5}]

## Moving the channels

In [13]:
await prep.pipettes.move_to_safe_z()

In [14]:
# The channels ride one gantry: they share x, channel 0 sits behind channel 1, and neighbours
# must stay at least 9 mm apart. Naming only channel 0 leaves channel 1 where it is, which the
# ordering check refuses, so send both.
await prep.pipettes.move_to_location(
    [Coordinate(150.0, 210.0, 100.0), Coordinate(150.0, 200.0, 100.0)],
    use_channels=[0, 1],
    make_space=True
)
# await prep.pipettes.sense_tip_presence()  # reads each channel's sleeve sensor; the chatterbox has none, so this raises in simulation

## Probe Core Grippers

In [15]:
# start_time = time.time()

# gripper = prep.get_resource("Prep_core_gripper_tool_back")

# gripper_center = gripper.get_location_wrt(deck, "c","c","t")

# results = await prep.pipettes.probe_edges_using_clld(
#     channel_idx=0,
#     target=gripper_center,
#     allow_without_tip=True
# )

# end_time = time.time() - start_time

# results, end_time

## CORE Gripper Pickup/Drop

In [16]:
# await prep.core_grippers.return_tools()

In [17]:
# await prep.core_grippers.return_tools()

## The X arm the channels ride

In [18]:
print("x:", await prep.x_arm.request_position())
print("commanded x:", await prep.x_arm.request_commanded_position())
print("speed:", await prep.x_arm.request_speed())
print("acceleration:", await prep.x_arm.request_acceleration())

x: 150.0
commanded x: 149.81
speed: 400.0
acceleration: 2250.0


## The 8-channel head

In [19]:
if prep.head8 is not None:
    await prep.head8.move_to_position(150.0, 200.0, 100.0)
    print(await prep.head8.sense_tip_presence())

[False, False, False, False, False, False, False, False]


## Tip Pickup

In [20]:
import time
from tqdm.auto import tqdm

from pylabrobot.resources import Coordinate, set_tip_tracking
from pylabrobot.resources.hamilton import hamilton_96_tiprack_50uL_NTR, hamilton_96_tiprack_300uL_NTR
set_tip_tracking(enabled=True)  # a spot gives up its tip, and takes it back

In [ ]:
from pylabrobot.resources import azenta_96_wellplate_200uL_Vb_4titudeframestar
from pylabrobot.resources import cor_96_wellplate_360ul_Fb, ResourceStack

from pylabrobot.byonoy import byonoy_sbs_adapter, byonoy_a96a_parking_unit, byonoy_a96a_detection_unit, byonoy_a96a_illumination_unit

In [ ]:
# Plates
measurement_plate_stack = deck[5] = 

measurement_plate_stack = deck[7] = ResourceStack(
    name="measurement_plate_stack",
    direction="z"
    )

for idx in range(4):
    plate = cor_96_wellplate_360ul_Fb(
        name=f"measurement_plate_{idx}",
        with_lid=True
        )
    measurement_plate_stack.assign_child_resource(plate)

# Tipracks

ntr_50ul_00 = deck[1] = hamilton_96_tiprack_50uL_NTR(name="ntr_50ul_00", with_tips=True)

ntr_300ul_00 = deck[6] = hamilton_96_tiprack_300uL_NTR(name="ntr_300ul_00", with_tips=True)

/var/folders/_3/0_4lk1cx741663czqtjw50qh0000gq/T/ipykernel_91561/3951330385.py:4: DeprecationWarning: Cor_96_wellplate_360ul_Fb() is deprecated and will be removed in v1b1. Use cor_96_wellplate_360uL_Fb() instead.
  measurement_plate_0 = deck[4] = Cor_96_wellplate_360ul_Fb(name="measurement_plate_0", with_lid=True)


In [ ]:
# import math

# from pylabrobot.resources import PetriDish

# # A cylinder of the model's 77 mm diameter (the inner diameter is not measured); uL = mm3
# GLASS_AREA = math.pi * (77.0 / 2) ** 2

# glass = PetriDish(
#     name="whiskey_glass",
#     diameter=77.0,
#     height=86.0,
#     material_z_thickness=11.0,
#     compute_volume_from_height=lambda h: h * GLASS_AREA,
#     compute_height_from_volume=lambda v: v / GLASS_AREA,
# )
# deck[6].assign_child_by_anchor(
#     resource=glass,
#     parent_anchor = ("c", "c", "t"),
#     child_anchor = ("c", "c", "b"),
#     )

2026-09-28 01:39:05,735 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'whiskey_glass' stands 99.5 mm high on the deck, above the 75.0 mm that is safe under any tip.


In [24]:
await prep.head8.move_to_safe_z()

167.5

In [25]:
byonoy_sbs_adapter_1 = deck[2] = byonoy_sbs_adapter(name="byonoy_sbs_adapter_1")
byonoy_sbs_adapter_0 = deck[0] = byonoy_sbs_adapter(name="byonoy_sbs_adapter_0")

a96a_parking_unit = byonoy_a96a_parking_unit(name="a96a_parking_unit")
a96a_detection_unit = byonoy_a96a_detection_unit(name="a96a_detection_unit")

byonoy_sbs_adapter_1.assign_child_resource(
    a96a_parking_unit,
    location=Coordinate(
        x=-23.5,
        y=-(95.48 - 85.48) / 2,
        z=17.0,
        )
    )
byonoy_sbs_adapter_0.assign_child_resource(
    a96a_detection_unit,
    location=Coordinate(
        x=-23.5,
        y=-(95.48 - 85.48) / 2,
        z=17.0,
        )
    )


In [26]:
a96a_illumination_unit = byonoy_a96a_illumination_unit(name="a96a_illumination_unit")

a96a_detection_unit.illumination_unit_holder.assign_child_resource(a96a_illumination_unit)

2026-09-28 01:39:05,776 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'a96a_illumination_unit' stands 77.498 mm high on the deck, above the 75.0 mm that is safe under any tip.


In [27]:
await prep.core_grippers.pick_up_tools()


In [27]:
await prep.core_grippers.pick_up_resource(
    resource=a96a_illumination_unit,
    pickup_distance_from_top=15.4,
    # y_clearance=2.5
)

await prep.core_grippers.drop_resource(
    to=a96a_parking_unit.illumination_unit_holder,
    z_acceleration=50
) 

2026-09-27 22:21:44,801 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'a96a_illumination_unit' stands 77.498 mm high on the deck, above the 75.0 mm that is safe under any tip.


In [28]:
await prep.core_grippers.move_resource(
    resource=measurement_plate_0,
    to=a96a_detection_unit.plate_holder,
    pickup_distance_from_top=12.0,
    drop_z_acceleration=50,
    return_grippers=False,
    # y_clearance=2.3
)
await prep.core_grippers.probe_resource_exists_using_ztouch(
    measurement_plate_0
)   

2026-09-27 22:22:51,553 - pylabrobot.hamilton.prep.driver.features.core_grippers - INFO - 'measurement_plate_0' found at 49.46 mm, its top at 50.70 mm


True

In [ ]:
# await prep.core_grippers.pick_up_resource(
#     resource=measurement_plate_0,
#     pickup_distance_from_top=12.0,
#     y_clearance=2.3
# )
# await prep.core_grippers.drop_resource(
#     to=a96a_detection_unit.plate_holder,
#     z_acceleration=50
# )

In [30]:
# await prep.core_grippers.pick_up_resource(
#     resource=a96a_illumination_unit,
#     pickup_distance_from_top=15.4,
#     y_clearance=2.5
# )

# await prep.core_grippers.drop_resource(
#     destination=a96a_detection_unit.illumination_unit_holder,
#     z_acceleration=50
# )

await prep.core_grippers.move_resource(
    resource=a96a_illumination_unit,
    to=a96a_detection_unit.illumination_unit_holder,
    pickup_distance_from_top=15.4,
)
await prep.core_grippers.probe_resource_exists_using_ztouch(
    a96a_illumination_unit
)   

2026-09-27 22:23:31,795 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'a96a_illumination_unit' stands 77.498 mm high on the deck, above the 75.0 mm that is safe under any tip.
2026-09-27 22:23:38,496 - pylabrobot.hamilton.prep.driver.features.core_grippers - INFO - 'a96a_illumination_unit' found at 75.76 mm, its top at 77.50 mm


True

In [32]:
await prep.core_grippers.pick_up_resource(
    resource=a96a_illumination_unit,
    pickup_distance_from_top=15.4,
    y_clearance=2.5
)

await prep.core_grippers.drop_resource(
    to=a96a_parking_unit.illumination_unit_holder,
    z_acceleration=50
)

2026-09-27 22:24:10,846 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'a96a_illumination_unit' stands 77.498 mm high on the deck, above the 75.0 mm that is safe under any tip.


In [33]:
await prep.core_grippers.pick_up_resource(
    resource=measurement_plate_0,
    pickup_distance_from_top=12.0,
)
await prep.core_grippers.drop_resource(
    to=deck[4],
    z_acceleration=50
)

In [34]:
await prep.core_grippers.pick_up_resource(
    resource=a96a_illumination_unit,
    pickup_distance_from_top=15.4,
    y_clearance=2.5
)

await prep.core_grippers.drop_resource(
    to=a96a_detection_unit.illumination_unit_holder,
    z_acceleration=50
)

2026-09-27 22:24:53,182 - pylabrobot.resources.hamilton.prep_decks - WARNING - Resource 'a96a_illumination_unit' stands 77.498 mm high on the deck, above the 75.0 mm that is safe under any tip.


In [35]:
await prep.core_grippers.return_tools()

In [ ]:
await prep.pipettes.discard_tips()

In [ ]:
# await prep.core_grippers.return_resource(y_clearance=2.5)

In [ ]:
ntr_50ul_00

In [ ]:
await prep.pipettes.pick_up_tips(
    # [prep.get_resource("Prep_teaching_needle")],
    ntr_50ul_00["A3:B3"],
    # use_channels=[1]
    )


In [ ]:
await prep.pipettes.return_tips()

In [ ]:
await prep.pipettes.pick_up_tips(
    # [prep.get_resource("Prep_teaching_needle")],
    ntr_50ul_00["A2:B2"],
    # use_channels=[1]
    )


In [ ]:
await prep.pipettes.drop_tips(ntr_50ul_00["A12:B12"])

In [ ]:
[t.unassign_tip() for t in ntr_50ul_00["A12:B12"]]

In [ ]:
await prep.pipettes.drop_tips(ntr_50ul_00["A12:B12"])

In [ ]:
await prep.pipettes.pick_up_tips(
    # [prep.get_resource("Prep_teaching_needle")],
    ntr_50ul_00["A1:B1"],
    # use_channels=[1]
    )


In [ ]:
await prep.pipettes.pick_up_tips(
    # [prep.get_resource("Prep_teaching_needle")],
    ntr_50ul_00["A12:B12"],
    # use_channels=[1]
    )


In [ ]:
await prep.pipettes.discard_tips()

In [ ]:
await prep.pipettes.pick_up_tips(
    # [prep.get_resource("Prep_teaching_needle")],
    ntr_300ul_00["A12:B12"],
    # use_channels=[1]
    )


In [ ]:
await prep.pipettes.return_tips()


In [ ]:
await prep.pipettes.pick_up_tips(
    # [prep.get_resource("Prep_teaching_needle")],
    [ntr_300ul_00["A12"][0], ntr_50ul_00["A4"][0]],
    # use_channels=[1]
    )


# Byonoy

In [ ]:
from pylabrobot.byonoy import byonoy_a96a
reader, illumination_unit = byonoy_a96a(name="a96a")
await reader.setup()
print(reader.available_wavelengths)

In [ ]:
print(reader.available_wavelengths)
print(await reader.request_device_info())
print(await reader.request_status())
print(await reader.request_environment())
print(await reader.request_versions())

In [ ]:
await reader.set_led_color((0, 255, 0))


In [ ]:
await reader.set_led_color((0, 0, 255), effect="breathing")


In [ ]:
await reader.set_led_color((0, 0, 0))

In [ ]:
supported = await reader.request_supported_reports()
print([hex(r) for r in supported])
print("LED bar supported:", 0x0350 in supported or 0x0351 in supported)

In [ ]:
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

reader.illumination_unit_holder.unassign_child_resource(illumination_unit)
plate = Cor_96_wellplate_360ul_Fb(name="plate")
reader.plate_holder.assign_child_resource(plate)
reader.illumination_unit_holder.assign_child_resource(illumination_unit)

In [ ]:
import pandas as pd

results = await reader.read_absorbance(plate, wells=plate.get_all_items(), wavelength=420)
pd.DataFrame(results[0].data, index=list("ABCDEFGH"), columns=range(1, 13)).round(3)

In [ ]:
2026-09-22 01:43:24,811 - pylabrobot.byonoy.absorbance_96 - INFO - [a96a] reading absorbance: plate='plate', wavelength=420 nm, wells=96/96
2026-09-22 01:43:26,508 - pylabrobot.byonoy.driver - WARNING - [a96a] non-zero chunk flags during read: ['0x01', '0x01', '0x01', '0x01', '0x01', '0x01', '0x01', '0x01'] (vendor bit definitions not published; data may be unreliable)
1	2	3	4	5	6	7	8	9	10	11	12
A	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
B	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
C	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
D	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
E	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
F	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
G	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0
H	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0


In [ ]:
await prep.pipettes.sense_tip_presence()

In [ ]:
await prep.pipettes.return_tips()

In [ ]:
target_cct = prep.get_resource("elution_plate_0").get_location_wrt(deck,"c","c","t") 
channel_idx=0

await prep.pipettes.move_to_safe_z()

await prep.pipettes.move_to_xy_positions(
        x=target_cct.x+8*9,
        ys={channel_idx:target_cct.y},
        # minimum_traverse_height_start=probing_target_coordinate.z,
        make_space=True
        )

z_results = []
n = 3

search_starting_z = 30
for _ in range(n):
    z_measurement = await prep.pipettes.probe_z_using_ztouch(
        channel_idx=0,
        search_start_position=search_starting_z,
        )
    z_results.append(z_measurement)
    search_starting_z = z_measurement + 5

await prep.pipettes.move_to_safe_z()

z_results

In [ ]:
prep.driver.x_arm.configuration

In [ ]:
channel_idx = 1 

await prep.pipettes.move_to_safe_z()

corner_offset_dict = {}

for x in [30, 235]:

        for y in [-2.5, 375]:

                await prep.pipettes.move_to_xy_positions(
                        x=x,
                        ys={channel_idx:y},
                        # minimum_traverse_height_start=probing_target_coordinate.z,
                        make_space=True
                        )

                z_results = []
                n = 3

                search_starting_z = 30
                for _ in range(n):
                        z_measurement = await prep.pipettes.probe_z_using_ztouch(
                                channel_idx=channel_idx,
                                search_start_position=search_starting_z,
                                )
                        if z_measurement is not None:
                                z_results.append(z_measurement)
                                search_starting_z = z_measurement + 5

                corner_offset_dict[f"{x},{y}"] = round(sum(z_results)/len(z_results), 2)

await prep.pipettes.move_to_safe_z()

corner_offset_dict

In [ ]:
corner_offset_dict

In [ ]:
start_time = time.time()

channel_idx=0

calibration_block_cct = prep.get_resource("Prep_calibration_block").get_location_wrt(deck,"c","c","t") 

await prep.pipettes.move_to_safe_z()

await prep.pipettes.move_to_xy_positions(
        x=calibration_block_cct.x,
        ys={channel_idx:calibration_block_cct.y},
        # minimum_traverse_height_start=probing_target_coordinate.z,
        make_space=True
        )

z_results = []
n = 3

search_starting_z = 30
for _ in range(n):
    z_measurement = await prep.pipettes.probe_z_using_ztouch(
        channel_idx=0,
        search_start_position=search_starting_z,
        )
    z_results.append(z_measurement)
    search_starting_z = z_measurement + 5

await prep.pipettes.move_to_safe_z()
z_result = round(sum(z_results)/len(z_results), 2)


results = await prep.pipettes.probe_edges_using_clld(
    channel_idx=channel_idx,
    target=Coordinate(calibration_block_cct.x, calibration_block_cct.y, z_result),
    n_replicates=n
)

# end_time = time.time() - start_time

calibration_block_cct, results, z_result

In [ ]:
184.06+(190.09-184.0)/2

In [ ]:


offset = Coordinate(-4, -2+0, 0)

target_coordinate = calibration_block_cct + offset

results = await prep.pipettes.probe_edges_using_clld(
    channel_idx=channel_idx,
    target=target_coordinate,
    n_replicates=5
)

end_time = time.time() - start_time

calibration_block_cct, results, end_time

In [ ]:
await 

In [ ]:
await prep.pipettes.request_locations()  # each channel's reference point, in deck mm

In [ ]:
await prep.pipettes.move_to_safe_z([0])
print("locations:  ", await prep.pipettes.request_locations())
print("z positions:", await prep.pipettes.request_z_positions())
print("stop disc:  ", await prep.pipettes.request_stop_disc_z_position(0))
print("z drive:    ", await prep.pipettes.channels[0].request_z_drive_position())
print("held length:", await prep.pipettes.request_held_tip_length())

In [ ]:
channel_idx = 1
n=3

plate_holders = [prep.deck.children[x] for x in [0, 3, 4, 7]]

ztouch_measurements = {}

start_time = time.time()

for idx, ph in enumerate(tqdm(plate_holders)):
    ccb_ph = ph.get_location_wrt(deck, "c", "c")+Coordinate(-65,0.,0)


    resp = await prep.pipettes.move_to_xy_positions(
        x=ccb_ph.x,
        ys={channel_idx:ccb_ph.y},
        # minimum_traverse_height_start=probing_target_coordinate.z,
        make_space=True
        )

    search_initial_z_position = 15
    tech_replicates = []
    for _ in range(n):
        resp = await prep.pipettes.probe_z_using_ztouch(
            channel_idx,
            search_start_position=search_initial_z_position,
            )#, allow_without_tip=True)
        tech_replicates.append(resp)
        search_initial_z_position = resp + 5

    await prep.pipettes.move_to_safe_z()

    ztouch_measurements[idx] = tech_replicates

end_time = round(time.time() - start_time, 2)

ztouch_measurements, end_time

In [ ]:
predestal_deck_surface = ({0: [-0.46, -0.44, -0.43],
  1: [-0.59, -0.6, -0.6],
  2: [-0.7, -0.7, -0.69],
  3: [-0.61, -0.63, -0.65]},
 34.58

In [ ]:
predestal_top = (
    {0: [4.12, 4.12, 4.13],
  1: [4.17, 4.18, 4.18],
  2: [4.13, 4.13, 4.13],
  3: [4.23, 4.23, 4.23]},
 31.82)

In [ ]:
4.17 - 3.25

In [ ]:
probed_corner_of_plate_holder = (
{0: [3.25, 3.22, 3.24],
  1: [3.24, 3.23, 3.25],
  2: [3.31, 3.31, 3.32],
  3: [3.44, 3.43, 3.44]},
 35.25)

In [ ]:
probed_NTRs = (
    {0: [58.23, 58.21, 58.21],
  1: [58.24, 58.24, 58.24],
  2: [58.27, 58.27, 58.27],
  3: [58.42, 58.42, 58.42]},
 46.05)

In [ ]:
prep.get_resource("tips").get_location_wrt(deck,z="t")

In [ ]:
({0: [4.225534057617182, 4.220361938476572, 4.230915527343768],
  1: [4.450814819335932, 4.445642700195322, 4.460915527343758],
  2: [4.204080200195307, 4.198908081054697, 4.2001885986328205],
  3: [4.890649414062494, 4.885477294921884, 4.890188598632818]},
 34.409687995910645)

In [ ]:
start_time = time.time()


a1_coordinate = prep.get_resource("tips")["A12"][0].get_location_wrt(deck,"c","c","t") 

results = await prep.pipettes.probe_edges_using_clld(
    channel_idx=1,
    target=a1_coordinate + Coordinate(-3,0,8),
)

end_time = time.time() - start_time

results, end_time

In [ ]:
start_time = time.time()


a1_coordinate = prep.get_resource("tips")["H1"][0].get_location_wrt(deck,"c","c","t") 

results = await prep.pipettes.probe_edges_using_clld(
    channel_idx=1,
    target=a1_coordinate + Coordinate(-3,0,8),
)

end_time = time.time() - start_time

results, end_time

In [ ]:
a1_coordinate

In [ ]:
prep.get_resource("Prep_spot_0_1").get_location_wrt(deck,"c","c","t") 

## Probing calibration block

In [ ]:
calibration_block_cct = prep.get_resource("Prep_spot_0_1").get_location_wrt(deck,"c","c","t") 
channel_idx=1

offset = Coordinate(-4, -1+0, 0)

await prep.pipettes.move_to_xy_positions(
    x=calibration_block_cct.x + offset.x,
    ys={channel_idx:calibration_block_cct.y + offset.y},
    # minimum_traverse_height_start=calibration_block_cct.z + offset.z,
    make_space=True

    )

await prep.pipettes.move_tool_bottom_to_z_positions({1: -2})

# back_measurements = []
# for _ in range(1):
#     resp = await prep.pipettes.probe_y_using_clld(
#         channel_idx=channel_idx,
#         direction="forward",
#         # search_start_position=y_start,
#         search_end_position=93,
#         allow_without_tip=True,
#         speed=5
#     )
#     back_measurements.append(resp)

back_measurements = []

y_start = calibration_block_cct.y

for _ in range(3):
    resp = await prep.pipettes.probe_y_using_clld(
        channel_idx=channel_idx,
        direction="backward",
        search_start_position=y_start,
        search_end_position=188,
        allow_without_tip=True,
        speed=5
    )
    back_measurements.append(resp)
    y_start = resp - 5

    if resp == None:
        y_start = calibration_block_cct.y
        await prep.pipettes.move_to_xy_positions(
            x=calibration_block_cct.x,
            ys={channel_idx:y_start},
            # minimum_traverse_height_start=140,
            make_space=True
        )


back_measurements



In [ ]:
# The same probe in X. Prep.glb draws each holder's aperture 112.00 x 77.00 mm, four per column,
# pitch 140.0 and 95.0 - so centred in a 133.5 x 91.5 holder it is deck x 10.75..122.75,
# y 102.25..179.25 for spot 0_1.
#
# Y already measured 101.32 / 178.14: centre 139.73, span 76.82. The 0.18 mm it falls short of the
# drawn 77.00 is the cLLD triggering 0.09 mm before each face - the calibration block read 0.18 mm
# over on a solid 6.00, the same gap with the sign reversed.

spot = prep.get_resource("Prep_spot_0_1")
spot_cct = spot.get_location_wrt(deck, "c", "c", "t")
channel_idx = 1

hole_size_x = 112.0        # as drawn in Prep.glb
hole_y_center = 139.73     # measured: (101.32 + 178.14) / 2

# Stand inside the hole, at an X already proven open, and drop below the deck once. Every search
# below leaves the channel backed off inside the hole, so no repeat needs a new X and a blind drop
# - which is what stalled the Z drive when a step went the wrong way.
await prep.pipettes.move_to_xy_positions(
    x=spot_cct.x - 4,
    ys={channel_idx: hole_y_center},
    make_space=True,
)
await prep.pipettes.move_tool_bottom_to_z_positions({channel_idx: -2})

slot_0_1_hole_left = []
for _ in range(3):
    slot_0_1_hole_left.append(
        await prep.pipettes.probe_x_using_clld(
            channel_idx=channel_idx,
            direction="left",
            search_end_position=2.5,    # wall expected near 6.4, so ~4 mm of travel past it
            allow_without_tip=True,
        )
    )

# Back to the middle of the hole before turning round, staying at the depth we are already at.
await prep.x_arm.move_to_x_position(spot_cct.x - 4, minimum_traverse_height_start=-2.0)

slot_0_1_hole_right = []
for _ in range(3):
    slot_0_1_hole_right.append(
        await prep.pipettes.probe_x_using_clld(
            channel_idx=channel_idx,
            direction="right",
            search_end_position=122.5,  # wall expected near 118.4
            allow_without_tip=True,
        )
    )

left = [v for v in slot_0_1_hole_left if v is not None]
right = [v for v in slot_0_1_hole_right if v is not None]
if left and right:
    lo, hi = sum(left) / len(left), sum(right) / len(right)
    print(f"hole x {lo:.2f} .. {hi:.2f}   span {hi - lo:.2f} (drawn {hole_size_x:.2f})")
    print(f"centre {(lo + hi) / 2:.2f}   model {spot_cct.x:.2f}   dx {(lo + hi) / 2 - spot_cct.x:+.2f}")

slot_0_1_hole_left, slot_0_1_hole_right


In [ ]:
back_measurements

In [ ]:
slot_0_1_hole_front = [101.33, 101.33, 101.29]

slot_0_1_hole_back = [178.14, 178.14, 178.14]

size_y = 178.14-101.31

In [ ]:
calibration_block_cct.y + calibration_block_cct.y - 93

In [ ]:
prep.get_resource("Prep_spot_0_1").get_location_wrt(deck,"c","f","t") 

In [ ]:
back_measurements = []

y_start = 100

for _ in range(3):
    resp = await prep.pipettes.probe_y_using_clld(
        channel_idx=channel_idx,
        direction="forward",
        search_start_position=y_start,
        search_end_position=93,
        allow_without_tip=True,
        speed=5
    )
    back_measurements.append(resp)
    y_start = resp + 5

    if resp == None:
        y_start = 100
        await prep.pipettes.move_to_xy_positions(
            x=calibration_block_cct.x,
            ys={channel_idx:y_start},
            # minimum_traverse_height_start=140,
            make_space=True
        )


back_measurements

In [ ]:
await prep.pipettes.request_locations()

In [ ]:
calibration_block_cct = prep.get_resource("Prep_calibration_block").get_location_wrt(deck,"c","c","t") 
channel_idx=1

offset = Coordinate(-4, -1+0, 0)

await prep.pipettes.move_to_xy_positions(
    x=calibration_block_cct.x + offset.x,
    ys={channel_idx:calibration_block_cct.y + offset.y},
    # minimum_traverse_height_start=calibration_block_cct.z + offset.z,
    make_space=True

    )

In [ ]:
search_initial_z_position = 30
tech_replicates = []
for _ in range(n):
    resp = await prep.pipettes.probe_z_using_ztouch(
        channel_idx,
        search_start_position=search_initial_z_position,
        )#, allow_without_tip=True)
    tech_replicates.append(resp)
    search_initial_z_position = resp + 5

calibration_top_measured = round(sum(tech_replicates)/len(tech_replicates), 2)
calibration_top_measured

In [ ]:
offset = Coordinate(-4, -1+10, 0)

await prep.pipettes.move_to_xy_positions(
    x=calibration_block_cct.x + offset.x,
    ys={channel_idx:calibration_block_cct.y + offset.y},
    # minimum_traverse_height_start=calibration_block_cct.z + offset.z,
    make_space=True

    )

await prep.pipettes.move_tool_bottom_to_z_positions({1: 21})

In [ ]:
back_measurements = []
for _ in range(1):
    resp = await prep.pipettes.probe_y_using_clld(
        channel_idx=channel_idx,
        direction="forward",
        # search_start_position=y_start,
        search_end_position=190,
        allow_without_tip=True,
        speed=5
    )
    back_measurements.append(resp)

    # if resp == None:
    #     await prep.pipettes.move_to_xy_positions(
    #         x=calibration_block_cct.x,
    #         ys={channel_idx:y_start},
    #         # minimum_traverse_height_start=140,
    #         make_space=True
    #     )
resp

In [ ]:
start_time = time.time()


calibration_block_cct = prep.get_resource("Prep_calibration_block").get_location_wrt(deck,"c","c","t") 
channel_idx=1

offset = Coordinate(-4, -2+0, 0)

target_coordinate = calibration_block_cct + offset

results = await prep.pipettes.probe_edges_using_clld(
    channel_idx=channel_idx,
    target=target_coordinate,
    n_replicates=5
)

end_time = time.time() - start_time

calibration_block_cct, results, end_time

In [ ]:
calibration_block_cct,prep.get_resource("Prep_calibration_block").get_location_wrt(deck,"c","c","t") 

In [ ]:
tech_replicates

In [ ]:
# await prep.pipettes.move_stop_disc_to_z_positions({1: 100})

In [ ]:
await prep.pipettes.move_to_safe_z()

In [ ]:
await prep.pipettes.return_tips()


In [ ]:
await prep.pipettes.probe_z_using_clld(1, search_start_position = 30)

In [ ]:
await prep.pipettes.probe_z_using_clld(0, allow_without_tip=True)

In [ ]:
await prep.pipettes.probe_z_using_clld(0, allow_without_tip=True)

## Probing: where a surface is

In [ ]:
# Lowers where the channel stands until the Z drive's following error builds. Heights are of the
# stop disc without a tip.
await prep.pipettes.probe_z_using_ztouch(0, allow_without_tip=True)

In [ ]:
await prep.pipettes.probe_z_using_clld(0, allow_without_tip=True)

In [ ]:
# cLLD in the raw: detection on for the enclosed block, then ask whether it triggered.
channel = prep.pipettes.channels[0]
async with channel.clld_detection(detect_mode=1, sensitivity=3):
    await prep.pipettes.move_to_location(
        [Coordinate(150.0, 210.0, 120.0), Coordinate(150.0, 200.0, 120.0)],
        use_channels=[0, 1],
    )
    print(await channel.request_clld_detected())

## The device itself

In [ ]:
# V3.0.20 answers this; the V1.2.2 recording the simulator speaks from has no IsParked method,
# so this raises PrepMethodNotFoundError in simulation.
await prep.driver.is_parked()

In [ ]:
await prep.driver.park_device()

## The deck light

`prep.lights` exists only on a device whose firmware declares one: that is what setup asks before
building it. A colour is a name, a hex string, `(r, g, b)` or `(white, r, g, b)`, and `brightness`
is a percentage, 0 to 100, of all four channels.

In [ ]:
print("installed:", await prep.driver.request_deck_light_installed())
print("standing at:", await prep.lights.request_color())  # (white, red, green, blue), each 0 to 255

In [ ]:
await prep.lights.set_color("magenta")

In [ ]:
await prep.lights.set_color("#00aaff", brightness=30)

In [ ]:
await prep.lights.turn_on(brightness=100)  # the white LEDs

In [ ]:
await prep.lights.turn_off()  # (0, 0, 0, 0)

### Animations

Each animation runs in the background, so the call returns as soon as the light is moving and the
protocol carries on. `turn_off`, any `set_color`, and `stop_animation` end one.

In [ ]:
# Red swelling into orange and back every 2 seconds, 30 to 100 percent of how bright it
# looks, until it is stopped.
await prep.lights.animate_error_pulse()

In [ ]:
print("mid-pulse:", await prep.lights.request_color())  # somewhere between red and orange

In [ ]:
await prep.lights.turn_off()  # ends the pulse and darkens the deck

In [ ]:
# Given seconds, the pulse ends itself with the light off.
await prep.lights.animate_error_pulse(duration=6)

In [ ]:
await prep.lights.disco()  # easter egg: random colours, then the colour it found

## Where the deck's own features are

The waste block is anchored by measurement: the parked grip tools were probed at x 289.99, which is
what put the block where it is. Everything else on the deck - the eight plate holders, the
calibration block, the frame drawn under them - comes from the measured OBJ, zeroed on spot 0's
clip corner, and **that corner has never been probed**. `prep_decks.py` says so in a TODO.

A rack's A1 landing between A1 and A2 is about half a well pitch, 4.5 mm, and it is the same on
every spot - which is an anchor that is out, not a pitch that is wrong (the pitch is 140.00 in the
OBJ, exactly).

The calibration block is the feature to measure it against: 6 x 6 mm of steel, 22 mm tall, and the
model puts its top centre at deck (136.75, 188.25). Probe its four faces; the difference between
the measured centre and that is the correction every OBJ-derived feature needs.

In [ ]:
# Probe the calibration block's four faces and say what the deck's own frame is out by.
from pylabrobot.resources import Coordinate
from pylabrobot.resources.hamilton.prep_decks import PREP_CALIBRATION_BLOCK_LOCATION

block = prep.deck.calibration_block
model_centre = block.get_location_wrt(prep.deck, "c", "c")
top = block.get_location_wrt(prep.deck, "c", "c", "t")
print(f"the model says its 6 x 6 top is centred at ({model_centre.x:.2f}, {model_centre.y:.2f}), "
      f"top z {top.z:.2f}")

await prep.pipettes.move_to_safe_z()
edges = await prep.pipettes.probe_edges_using_clld(
    channel_idx=0,
    target=Coordinate(model_centre.x, model_centre.y, top.z),
    n_replicates=2,
    allow_without_tip=True,   # the stop disc does the touching
    below_the_top=3.0,        # into the 6 x 6 part, above where it flares out at 14 mm
)
print(edges)

left, right = [e for e in edges["left"] if e is not None], [e for e in edges["right"] if e is not None]
front, back = [e for e in edges["front"] if e is not None], [e for e in edges["back"] if e is not None]
measured_x = (sum(left) / len(left) + sum(right) / len(right)) / 2
measured_y = (sum(front) / len(front) + sum(back) / len(back)) / 2
print(f"measured centre ({measured_x:.2f}, {measured_y:.2f})  "
      f"width {sum(right)/len(right) - sum(left)/len(left):.2f}  "
      f"depth {sum(back)/len(back) - sum(front)/len(front):.2f}   (6.00 x 6.00 expected)")
print(f"the deck's frame is out by dx {measured_x - model_centre.x:+.2f}  "
      f"dy {measured_y - model_centre.y:+.2f} mm")
print("half a well pitch is 4.50 mm; that is what a rack's A1 landing between A1 and A2 means")

# STOP

In [36]:
# try:
#     await prep.pipettes.return_tips()
# except Exception as e:
#     print(f"failed return of tips: {e}")

await prep.stop()

2026-09-27 22:25:12,218 - pylabrobot.io.socket - INFO - Closing connection to socket 192.168.100.102:2000
2026-09-27 22:25:12,219 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client stopped
2026-09-27 22:25:12,220 - pylabrobot.io.socket - INFO - Closing connection to socket 192.168.100.102:2000
2026-09-27 22:25:12,221 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client stopped


In [ ]:
await viewer.stop()

In [ ]:
await reader.stop()